In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
df = pd.read_csv('data/census.csv')

In [ ]:
placeholders = ['None', 'none', 'N/A', 'Unknown', 'unknown']
df.replace(placeholders, np.nan, inplace=True)

In [ ]:
df.head()


In [ ]:
df.info()

In [ ]:
# First Observation is that all the columns are strings (object type) including numeric ones like age
# The MArital Status, Infirmity and Religion contain missing values


In [ ]:
missing_percent = ( df.isnull().sum() / len(df) ) * 100
missing_percent

# Lets create a copy of the original data


In [ ]:
df_clean = df.copy()

# Lets clean and convert AGE column to numeric because people's AGE sholuld not Object (String)

In [ ]:
# checking for the unique value in the AGE Column
df_clean['Age'].unique()

In [ ]:
#convert the AGE column to numeric, coercing any invalid value entries to NAN
# Then round the numeric ages to the nearest whole number
#Finally, convert the result to an INTEGER
df_clean['Age'] = pd.to_numeric(df_clean['Age'], errors='coerce' ).round().astype('Int64')

In [ ]:
#checking if there are still any missing data points in the AGE column
df_clean[df_clean['Age'].isna()]

## SO it looks like there are still are rows with missing data points in the AGE column
we will use the Relationship to Head of House column to infer the age of the remaining people with missing age

In [ ]:
# Filter the DataFrame to include only rows where the person is the husband
#Then select the AGE column from those rows
# Finally , calculate the median age of the husbands
age_husband = df_clean[(df_clean['Relationship to Head of House'] == "Husband")]['Age'].median()

In [ ]:
#repeat the above steps for the 'SON'
age_son = df_clean[(df_clean['Relationship to Head of House'] =='Son')]['Age'].median()

In [ ]:
#repeat the above step for 'HEAD'
age_head = df_clean[(df_clean['Relationship to Head of House'] =='Head')]['Age'].median()

In [ ]:
#repeat the above steps for 'Daughter'
age_daughter = df_clean[(df_clean['Relationship to Head of House'] =='Daughter')]['Age'].median()

In [ ]:
#set the age of the record at index 8979 to the previoulsy calculated median age of husbands
df_clean.loc[8979, 'Age'] = age_husband

#set the age of the record at index 10272 to the median age of sons
df_clean.loc[10272, 'Age'] = age_son

# set the age of the record at the index 10351 to the median age of heads
df_clean.loc[10351, 'Age'] = age_head

#set the age of the record at index  10442 at the median age of daughter
df_clean.loc[10442, 'Age'] = age_daughter

# Lets clean the Gender column

In [ ]:
# checking for unique value in the gender column
df_clean['Gender'].unique()

### As we see here there is a bit of repetition , the character M logically will represent Male while F will represent Female

In [ ]:
# Remove leading space and capitalize the first letter of each value in the Gender column
df_clean['Gender'] = df_clean['Gender'].str.strip().str.title()

# Replace shorthand gender values : "M" with "MAlE" and "F" with "FEMALE"
df_clean['Gender'] = df_clean['Gender'].replace({ 'M': 'Male', 'F': "Female" })

In [ ]:
# Display the unique values present in the Gender column after cleaning
df_clean['Gender'].unique()

# Let's work on the Marital Status Column

In [ ]:
# checking for unique value in the Marital Status column
df_clean['Marital Status'].unique()

### The Marital Column has the same issue as the Gender column, [S, D, W, M] these values should represent Single , Divorced , Widowed, and Married respectivley Also it look like there is someone in our data that does not have any Martial Status

In [ ]:
# Lets check for that person without any Marital Status
df_clean.loc[df_clean['Marital Status'] == ' ']

### Looking at the person's age he is most likely to be SINGLE

In [ ]:
# Replave shorhand codes in the Marital Status column with full descriptions
# "M" - Married, 'S' - 'Single', 'D' - 'Divorced', 'W' - 'Widowed'
df_clean['Marital Status'] = df_clean['Marital Status'].replace({'M': "Married", 'S':'Single', 'D':'Divorced', 'W':'Widowed'}) 

### There are missing data points in the marital column so let's work on that
### So logically:
### -- People under the age of 18 likely are Single
### -- Spouses or Heads over 30 likely Married 
### -- Elderly individuals may be Widowed

In [ ]:
# Filter the Data to include only rows where both 'Marital Status' and 'Age' are not missing
valid_marital_age  = df_clean[   df_clean['Marital Status'].notna() & df_clean['Age'].notna()   ]

#create a box plot to visualize the distribution of 'Age' by 'Marital Status'
plt.figure(figsize=(6, 6))
sns.boxplot(data = valid_marital_age, x='Marital Status', y='Age')

#set the title
plt.title('Age Distributiion by Marital Status')

#Rotate the x-axis labels for better readbility
plt.xticks(rotation=45)

#display the plot
plt.show()

###  From the boxplot , we observe that
### . Single individuals cluster in lower age ranges (mostly under 30)
### . Married  individuals span broadly from late 20s to 60s
### . Widowed status generally corresponds with older ages (often over 60)
### . Divorced tands to fall in mid-age (30 - 60)

In [ ]:
# cretae a table (crosstab) to show the percentage distribution of 'Marital Status' by Relationship to Head of House
# Normalize the row, to show each marital status in each relationship
relationship_marital = pd.crosstab(valid_marital_age['Relationship to Head of House'], valid_marital_age['Marital Status'], normalize='index'  ) * 100

# cretae a table (crosstab) to show the percentage distribution of 'Marital Status' by Gender
# Normalize the row, to show each marital status in each relationship
gender_marital = pd.crosstab(valid_marital_age['Gender'], valid_marital_age['Marital Status'], normalize='index') * 100

In [ ]:
relationship_marital

### Key Insight from Correlation Analysis
### Husband/Wife: are exclusive marked Married
### Son/Daughter/GrandChild: Dominantly Single (70 - 80%)
### Head of house: Mixed but leans towards Married, with 9% Widowed
### Visitor/Lodger : Mostly Single With small portions Widowed
### Sibling : Mostly Single, Some Windowed


In [ ]:
gender_marital

### Gender Label Varies
### Males have higher proportions of Single And Married
### Females are more Likely to Widowed

### Lets define a rule for imputing missing value in Marital Status
### 1 Age:
### . Age < 18 = 'Single'
### . Age 60 + with no spouse in household = 'Widowed' 

### 2 Relationship:
### . Wife or Husband = Married
### . Son, Daughter, Visitor, Lodger = Single

### 3. Gender + Age (if Head):
### Female over 60 as Head = Likely Widowed
### Male over 60 with no spouse = Widowed

In [ ]:
# create a column called Gender_clean by converting all value in the gender column to lowercase
df_clean['Gender_clean'] = df_clean['Gender'].str.lower()

# create a new column called relationship_clean by converting all values to lowercase
df_clean['Relationship_clean'] = df_clean['Relationship to Head of House'].str.lower()

In [ ]:
# Define a function to clean the MArital Status based on specific condition
def clean_marital_status(row):
    #if the Marital Status is already not missing return it as it is
    if pd.notna(row['Marital Status']):
        return row['Marital Status']
    #Extract the values 'Age', 'Relationship' and 'Gender' from the current row
    age = row['Age']
    relationship = str(row['Relationship_clean']).lower().strip() if pd.notna(row['Relationship_clean']) else ''
    gender = row['Gender_clean'] if pd.notna(row['Gender_clean']) else ''

    #if A'Age is provided and person is under 18, classify them ae Single
    if pd.notna(age) and age < 18:
        return 'Single'
    #if the relationship indicates the person is a child, visitor or similar, classify them as SINGLE
    if relationship  in ['son', 'daughter', 'grandson', 'granddaughter', 'lodger', 'visitor',
                        'adopted son', 'adopted daughter', 'sibling', 'step-son', 'step-daughter']:
        return 'Single'

    #if the relationship indicates a married couple (husband/wife), return 'Married'
    if relationship in ['wife', 'husband']:
        return 'Married'
    #if the relationship is the head nad the person is 60 or older
    if relationship == 'head' and pd.notna(age) and age >= 60:
        #if the person is female, classify them as Widowed, otherwise classify them as Married
        if gender == 'female':
            return 'Widowed'
        else:
            return 'Married'
     #if the person is 60 or older and not wife or husband, classify as widowed   
    if pd.notna(age) and age >= 60 and relationship not in ['wife', 'husband']:
        return 'Widowed'
    #For all other cases, classify as Single
    return 'Single'

In [ ]:
# apply the Clean_marital_status function to each row in our data
df_clean['Marital Status'] = df_clean.apply(clean_marital_status, axis =1)

In [ ]:
df_clean.loc[df_clean['Marital Status'] == ' ', 'Marital Status'] = 'Single'

In [ ]:
#checking if there are still any missing values in the Marital Status column
df_clean['Marital Status'].isnull().sum()

# Lets move to the Religion column

In [ ]:
religion_counts = df_clean['Religion'].value_counts()
religion_counts

### Top Religions
### Christain = 2300
### Catholic 1144
### Muslim, Sikh, Jewish = smaller

In [ ]:
def fill_missing_religion(row, grouped_data):
    if pd.isna(row['Religion']):
        #creaate a tuple of grouping keys
        group_key = (row['House Number'], row['Street'])
        try:
            #get the not-null religions for this data
            house_group = grouped_data.get_group(group_key)
            religions = house_group['Religions'].dropna().unique()
            if len(religions) == 1:
                return religions[0]
            elif len(religions) > 1:
                return house_group['Religion'].mode()[0]
            else:
                return 'Unknown'
        except KeyError:
            #No group found for this address
            return 'Unknown'
    else:
        return row['Religion']
# Group b both House Number and Street
grouped_data = df_clean.groupby(['House Number', 'Street'])

In [ ]:
#Apply the function
df_clean['Religion'] = df_clean.apply(lambda row: fill_missing_religion(row, grouped_data), axis=1)

In [ ]:
df_clean.isnull().sum()

# Lets work on the Relationship Column

#### Infer famliy relationship based on age, gender and marital status. Returns the existing relationship if alreddy specified, otherwise infer


In [ ]:
def infer_relationship(age, gender, marital_status, existing_relationship):
    #if the relationship is already specified nd it is not empty, return it as it is
    if pd.notna(existing_relationship) and str(existing_relationship).strip() != '':
        return existing_relationship
    try:
        #convert the age to integer
        age = int(age)
    except (ValueError, TypeError):
        return 'Unknown'
    # convert strings  to lowercase 
    gender = gender.lower()
    martial_status = marital_status.lower()
    #infer based on age
    if age < 18:
        if gender == 'male':
            return 'Son'
        elif gender == 'female':
            return 'Daughter'
        else:
            return 'Child'
    if 18 <= age <= 25:
        if gender == 'male':
            return "Grandson"
        elif gender == 'female':
            return 'Grand daughter'
    #infer based on Marital Status
    if marital_status == 'married':
        if gender == 'male':
            return 'Husband'
        elif gender == 'female':
            return 'Wife'
    #Other options
    if gender == 'male':
        return 'Lodger'
    elif gender == 'female':
        return 'Visitor'
    return 'Other'

In [ ]:
#create a new column called Relationship to Head of House Clean and apply the function we just created
df_clean['Relationship to Head of House clean'] = df_clean.apply(lambda row: infer_relationship(row['Age'], row['Gender'], row['Marital Status'],
                                   row['Relationship to Head of House']), axis=1)

# Let's work on the Infirmity

In [ ]:
#checking for unique values in the Infirmity column
df_clean['Infirmity'].unique()

In [ ]:
#replace the missing data points : blank spaces and NAN values with 'Unknown'
df_clean['Infirmity'] = df_clean['Infirmity'].replace( ([' ', np.nan]), 'Unknown' )

In [ ]:
#drop uncleaned columns
df_clean.drop(['Relationship to Head of House', 'Relationship_clean'], inplace=True, axis=1)

In [ ]:
#drop the unclean Gender Column
df_clean.drop(['Gender'], inplace=True, axis=1)

In [ ]:
df_clean.isna().sum()

# Plots 

In [ ]:
#set the size of the figure
plt.figure(figsize=(8, 6))

#Create a histogram of Age column from from the cleaned Dataframe
sns.histplot(df_clean['Age'], bins=30, kde=True, color='skyblue')

#set the title of the plot
plt.title('Age Distribution')

# label the x-axis as age
plt.xlabel('Age')

#Label the y-axis as Count
plt.ylabel('Count')

#Display the plot
plt.show()

### Age distribution is across ages 0 to 105
### There is a notebale presence of middle aged individuals (40 -60), with notable populations of both young adults (18 -30)

## (a) What should be built on an unoccupied plot of land that the local government wishes to develop?

# Household Occupancy

In [ ]:
#Group the data by 'House Number' and 'Street' to identify unique households,
#then count the number of members in each household
households = df_clean.groupby(['House Number', 'Street']).size()

#Count how many households have a given number of people (i.e., household size)
#and sort the result by household size
household_size_distribution = households.value_counts().sort_index()

#Create a new figure with a specifc size for better readability
plt.figure(figsize=(8,5))

#Plot a bar chart of household size distribution
#x-axis: number of people in a household (converted to a string for better labeling)
#y-axis: number of households with that size
sns.barplot(x=household_size_distribution.index, y=household_size_distribution.values, palette='Set1', legend=False, hue=household_size_distribution)

#Set the title and the axis labels
plt.title('Household Size Distribution')
plt.xlabel('Number of People per Household')
plt.ylabel('Number of Household')

#Adjust the layout to prevent clipping of labels and titles
plt.tight_layout()

#Save the figure to a file with high resolution
plt.savefig('Household Distribution', bbox_inches='tight', dpi=300)

#Display the plot
plt.show()

#### IMPLICATION:
##### Most household have between 2 and 4 members, suggesting standard nuclear family units. A few outliers showing high occupancy, which possilbly is indicating shared housing or care homes. There is no evidence of significant over crowding or underuse . So i can say that this doesn't create a pressing case for housing

### Lets understand the occupation who are commuters

##### Many Occupations imply commuting because they require regular travels to the workplace, very few jobs are done remotely. For this analysis, we are going to look at the top 20 occupation to understanding if there is a large number of commuters in the community

In [ ]:
#Count the occurences of each unique value in the 'Occupation' column,
#sort them in the descending 0rder (most common first,)
#and display the top 20 most frequent occupations
df_clean['Occupation'].value_counts().head(20)

In [ ]:
#Create a new 'Age Group' column in the cleaned DataFrame by binning the 'Age' column
df_clean['Age Group'] = pd.cut(df_clean['Age'], bins = [0, 4, 17, 24, 44, 64, 74, 105], labels=['0-4 (Infants/Toddlers)', '5-17 (School Age)' ,'18-24 (Young Adults)', '25-44 (Prime Working Age) ', '45-64 (Mid-Career)', '65-74 (Early Retirement)', '75+ (Elderly)'], include_lowest=True)

In [ ]:
#Define a list of keywords that likely indicate commuter-related occupations 
commuter_keywords = ['university student', 'accountant' ,'consultant', 'analyst', 'advisor', 'officer',  'engineer', 'architect', 'lecturer', 'trainer', 'sales', 'lawyer', 'legal', 'bank', 'marketer']

#Filter the dataset for rows where the 'Occupation' column contains any of the commuter keywords
#Convert all text to lowercase for case-insensitive matching
#'na=False' avoids all errors if there are missing values (NaNs)
commuter_roles = df_clean[df_clean['Occupation'].str.lower().str.contains('|'.join(commuter_keywords), na=False)]

#Specifically extract all entries labeled 'university students'
university_students = df_clean[df_clean['Occupation'].str.lower().str.contains('university student')]

#Get the top 15 most common commuter-related occupations
top_commuter_roles = commuter_roles['Occupation'].value_counts().head(15)

#Create a horizontal bar chart to visualize the top commuter-related occupations
plt.figure(figsize=(10, 6))
sns.barplot(y=top_commuter_roles.index, x=top_commuter_roles.values, palette='muted', legend=False, hue=top_commuter_roles )

#Set the chart title and axis labels
plt.title('Top commuter-Linked Occupations')
plt.xlabel('Number of people')
plt.ylabel('Occupation')

#Optmize layout and save the figure as a PNG with high resolution
plt.tight_layout()
plt.savefig('Distribution of commuting occupations based on refined criteria', bbox_inches='tight', dpi=300)

#Display the plot
plt.show()

#Show the total number of identified commuters and the number of university students
len(commuter_roles), len(university_students)

In [ ]:
#Calculate the percentage of commuters in the cleaned dataset
commuter_percentage = (len(commuter_roles) / len(df_clean)) * 100

#Calculate the percentage of university students in the cleaned dataset
university_student_percent = (len(university_students) / len(df_clean)) * 100

#Estimate the total number of commuters by adding both percentages
#(assuming there's no overlap between commuters and university students)
total_number_of_commuters = commuter_percentage + university_student_percent

#Display the computed percentages and total
commuter_percentage,  university_student_percent, total_number_of_commuters


University Students: 2782 individuals are identified as students, including 714 explicitly labeled as university students, all of whom are commuters due to the absence of a university in town

Likely Commuters: 4469 people gave occupations srongly associated with commuting, such as enginers, consultants, and mangers

Top Occupations: Besides students, jobs like pension consultant, techinal support officer are common, indicating a notable amount of commuting workforce

This clearly supports the case for building a train station to ease pressure on the roads and support a growing commuter population

In [ ]:
#Get the top 10 most frequent occupations in the cleaned dataset
top_occupations = df_clean['Occupation'].value_counts().head(10)

#Extract just the occupation names from the index of the top occupations
top_occ_names = top_occupations.index.tolist()

#Filter the original Dataframe to only include the rows with those top 10 occupations
filtered_df = df_clean[df_clean['Occupation'].isin(top_occ_names)]

#Set the size of the plot
plt.figure(figsize =(12, 6))

#Create a horizontal count plot of occpations, segmented by gender
sns.countplot(data=filtered_df, y='Occupation', hue='Gender_clean', order=top_occ_names, palette='Set1')

#Add a title and axis labels to the plot
plt.title('Top Occupations by Gender')
plt.xlabel('Count')
plt.ylabel('Occupation')

#Save the plot as an image file
plt.savefig('Occupation by Gender')

#Display the plot
plt.show()

In [ ]:
#Count the number of occurrences of each religion in the cleaned dataset
religion_counts = df_clean['Religion'].value_counts()

#Calculate the percentage that each religion represents out of the total
religion_percent = (religion_counts / religion_counts.sum()) * 100

#Create a summary DataFrame with both count and percentage values for each religion
religion_summary = pd.DataFrame(
    {
        'Count': religion_counts,
        'Percentage': religion_percent.round(2)
    }
)

In [ ]:
#Set the size of the figure (width=6 inches, height=8 inches)
plt.figure(figsize=(6, 8))

#Create a vertical bar plot of the religion counts
religion_counts.plot(kind='bar')

#Label the x-axis as 'Occupation' 
plt.xlabel('Occupation')

#Label the y-axis to show it represents the number of people
plt.ylabel('Number of People')

#Rotate x-axis labels 45 degrees and align them to the right for readability 
plt.xticks(rotation=45, ha='right')

#Add a title to the plot
plt.title('Distribution of Religions')

#Automatically adjust the plot
plt.tight_layout()

#Display the plot
plt.show()

In [ ]:
#Display the religion summary DataFrame showing counts and percentages for each religion
religion_summary

In [ ]:
#Create a cross-tabulation table showing the count of each religion within each age group
religion_age = pd.crosstab(df_clean['Age Group'], df_clean['Religion'])

#Select only the top 6 most common religions based on the total count accross all age groups
#This is done by summing the columns (religions) and sorting them in descending order 
religion_age_ct = religion_age[religion_age.sum().sort_values(ascending=False).index[:6]]

#Display the resulting cross-tabulation table
religion_age_ct

In [ ]:
#Create a stacked bar chart from the cross-tabulated DataFrame
religion_age_ct.plot(kind='bar', stacked=True, figsize=(12, 8), colormap='coolwarm')

#Add a title to the plot
plt.title('Religion by Age Group')

#Label x-axis as 'Age Group'
plt.xlabel('Age Group')

#Label y-axis to represent the number of people
plt.ylabel('Number of people')

#Adjust layout to prevent overlapping elements
plt.tight_layout()

#Save the plot to a file with a high resolution(300 dpi) and tight bounding box
plt.savefig('Religion by age Group ',  bbox_inches='tight', dpi=300)

#Display the plot
plt.show()

A large portion of the town is listed as unknown, the town is dominated by christain and catholics, there are also a few Muslims,  a lot of the population religions is unknown

No clear demand for a second Catholic church
Moderate justification to consider a Methodist worship space
Muslim population might look for a warrant a small mosque

In [ ]:
#Count the number of individuals in each age group
#value_counts() returns counts; sort_index() ensures the age groups appear in logical order
age_group_counts = df_clean['Age Group'].value_counts().sort_index()

In [ ]:
#Group the data by Age group and Gender, count the number of occurences in each group
#'observed=True' ensures only observed combinations are returned (faster, cleaner)
age_group_gender_counts = df_clean.groupby(['Age Group', 'Gender_clean'], observed=True).size().reset_index(name='Count')

#Set the figure size for the plot
plt.figure(figsize=(10, 6))

#Create a grouped bar plot showing the count of people in each age group by gender
sns.barplot(data=age_group_gender_counts, x='Age Group', y='Count', hue='Gender_clean', palette='Set1')

#Add title and axis labels
plt.title('Age Distribution of the Town Population by Gender')
plt.xlabel('Age Group')
plt.ylabel('Number of People')

#Rotate x-axis labels for readability
plt.xticks(rotation=30, ha='right')

#Adjust layout to avoid the clipping of labels
plt.tight_layout()

#Save the figure as an image file with high resolution
plt.savefig('Age Distribution by Gender', bbox_inches='tight', dpi=300)

#Display the plot
plt.show()

In [ ]:
#Set the size of the plot (10 inches wide, 6 inches tall)
plt.figure(figsize=(10,6))

#Create a bar plot for age group counts
sns.barplot(x=age_group_counts.index, y=age_group_counts.values,  hue=age_group_counts, palette='Set1', legend=False)

#Add a title and axis labels
plt.title('Age Distribution of the Town Population')
plt.xlabel('Age Group')
plt.ylabel('Number of People')

#Rptate x-axis labels for better readability
plt.xticks(rotation=30, ha='right')

#Adjust layout to prevent label clipping 
plt.tight_layout()

#Save the plot as an image file
plt.savefig('Age Distribution of the ',  bbox_inches='tight', dpi=300)

#Display the plot
plt.show()

A strong presence in the 25-44 and also the 45 - 64 age group, indicating many working age adults 
The 18-24  group is sizeable , likely reflecting the presence og university students commuting from the town
The 0 - 17 range shows a moderate number of children and teens, suggesting  some growth in young families
The 65+ segments are smaller, indicating some need for retirement and healthcare planning
In conclusion, this is a youth skewed population

In [ ]:
#Count the number of individuals in each 'Infirmity' category
medical_needs_counts = df_clean['Infirmity'].value_counts()

#Calculate the percentage that each infirmity category represents
medical_needs_percent = (medical_needs_counts / medical_needs_counts.sum()) * 100

#Create a summary DaraFrame showing both count and percentage for each medical need category 
medical_summary = pd.DataFrame(
    {
        'Count': medical_needs_counts,
        'Percentage': medical_needs_percent.round(2)
    }
)

In [ ]:
medical_needs_counts

In [ ]:
#Create a cross-tabulation table showing the count of each occupation within age group
occupation_age_group = pd.crosstab(df_clean['Age Group'], df_clean['Occupation'])

#Filter the table to include only the top names (stored in top_occ_names)
occupation_age_group = occupation_age_group[top_occ_names]

#Plot the data as a stacked bar chart 
occupation_age_group.plot(kind='bar', stacked=True, figsize=(12, 7), colormap='Set1')

#Add a title and axis label to the plot 
plt.title('Occupation by Age Group')
plt.xlabel('Age Group')
plt.ylabel('Count')

#Display the plot 
plt.show()

#Display the underlying data table for refernce or further analysis
occupation_age_group

As we see here there are lot unemployed people between the ages of 25 - 44 and 45 - 64 agr groups, this indicates that the community has an Unemployment issue

There are a lot of students between the ages of 0 - 17 while 18 - 24 are mostly university students, that might also be about to graduate, thereby increasing the unemployment status of this community

There is s stable number of Retired individuals, that are nemployed, this is normal

This supports investment in employment and training programs
Also with this i can say that Low-density housing falls out, because with the rate of unemployment , it implies low-affluence

In [ ]:
#Mark the arrows where the occupation contains the word 'unemployed' (case-insensitive)
df_clean['Unemployed'] = df_clean['Occupation'].str.lower().str.contains('unemployed')

#Group the data by age group and calculate:
#-total number of people in each group
#-number of people marked as unemployed
grouped = df_clean.groupby('Age Group', observed=True).agg(
    total=('Occupation', 'count'),
    unemployed=('Unemployed', 'sum')
)

#Calculate umemployment rate as a percentage
grouped['Unemployment_Rate'] = (grouped['unemployed'] / grouped['total']) * 100

#Reset indec so 'Age Group' becomes a column again
grouped = grouped.reset_index()

In [ ]:
grouped

In [ ]:
#Count how many entries in occupation contain the word 'unemployed'
unemployed_count = df_clean['Occupation'].str.lower().str.contains('unemployed').sum()

#Total number of records in the cleaned dataset
total_count = len(df_clean)

#Calculate the overall unemployment percentage
unemployment_percentage = (unemployed_count / total_count) * 100

#Display the result
unemployment_percentage

In [ ]:
#Set the figure size for the plot
plt.figure(figsize=(8, 5))

#Create a barplot showing unemployment rate by age group 
sns.barplot(data=grouped, x='Age Group', y='Unemployment_Rate', hue='Age Group', palette='pastel', legend=False)

#Add a title and axis labels
plt.title('Trend of Unemployment Across age Groups')
plt.xlabel('Age Group')
plt.ylabel('Unemployment_Rate')

#Rotate x-axis labels for better readability
plt.xticks(rotation=30, ha='right')

#Adjust layout to prevent clipping
plt.tight_layout()

#Save the plot as a high resolution PNG file
plt.savefig('Unemployment', bbox_inches='tight', dpi=300)

#Display the plot
plt.show()

In [ ]:
#Set the size of the figure
plt.figure(figsize=(8, 6))

#Count the frequency of each marital status category
marital_counts = df_clean['Marital Status'].value_counts()

#Create a barplot using the index (categories) and their corresponding counts
sns.barplot(x=marital_counts.index, y=marital_counts.values, hue=marital_counts, palette='Set2', legend=False)

#Add a title and axis labels
plt.title('Marital Status Distribution')
plt.title("Marital Status Distribution")
plt.ylabel('Number of People')

#Rotate x-axis for better readability
plt.xticks(rotation=30, ha='right')

#Save the plot as a high resolution PNG
plt.savefig('Marital Status Distribution', bbox_inches='tight', dpi=300)

#Show the plot
plt.show()

### There are more Single people in our data

In [ ]:
#Group the data by 'Age Group' and 'Marital Status', then count the number in each group
age_group_marital_counts = df_clean.groupby(['Age Group', 'Marital Status'], observed=True).size().reset_index(name='Count')

#Set the figure size 
plt.figure(figsize=(10, 6))

#Create a grouped bar plot showing count of marital status within each age group
sns.barplot(data=age_group_marital_counts, x='Age Group', y='Count', hue='Marital Status', palette='Set1')

#Add title and axis labels 
plt.title('Age Distribution of the Town Population by Marital Status')
plt.xlabel('Age Group')
plt.ylabel('Number of People')

#Rotate x-axis for better readability
plt.xticks(rotation=30, ha='right')

#Adjust layout to avoid clipping
plt.tight_layout()

#Save the figure as a high-resolution PNG
plt.savefig('Age Distribution by Marital Status', bbox_inches='tight', dpi=300)

#Display the plot
plt.show()

In [ ]:
#Count the number of occurences of each marital status 
marital_counts = df_clean['Marital Status'].value_counts()

#Calculate the percentage of each marital status
marital_percent = (marital_counts / marital_counts.sum()) * 100

#Combine count and percentage into a summary Dataframe
marital_summary = pd.DataFrame({
    'Count': marital_counts,
    'Percentage': marital_percent.round(2)
})

#Display the summary table
marital_summary

In [ ]:
#Count how many individuals are aged 75 or older - assumed to be retired
retired = len(df_clean[df_clean['Age'] >= 75])

#Calculate the percent of individuals relative to the whole population
percent_retired = retired / len(df_clean) * 100

#Output the result
percent_retired

In [ ]:
#Count how many individuals are aged 75 or older - assumed to be retired
retired = len(df_clean[df_clean['Age'] >= 75])

#Calculate the percent of individuals relative to the whole population
percent_retired = retired / len(df_clean) * 100

#Output the result
percent_retired

The ageing population is about 10%, now this data is youth-skewed so we have to consider that also, even if the retired are more

In [ ]:
#Count the number of individuals aged from 0 to 4
new_births = len(df_clean[df_clean['Age'] <= 4])

#Calculate the percentage based on the cleaned dataset
new_birth_percent = new_births / len(df) * 100

#Display the result
new_birth_percent

In [ ]:
The number of new births are about 6% of the population, this does not strongly support Increase spending for schooling.